## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-14-18-53-48 +0000,nyt,Repair That Triggered Air Traffic Meltdown Was...,https://www.nytimes.com/2026/11/14/us/repair-t...
1,2026-11-14-18-47-05 +0000,startribune,Science Museum of Minnesota to cut 22 jobs ami...,https://www.startribune.com/science-museum-of-...
2,2026-11-14-18-44-13 +0000,nyt,How Does a 74-Year-Old in Argentina Escape Hou...,https://www.nytimes.com/2026/11/13/world/ameri...
3,2026-11-14-18-42-03 +0000,startribune,"Live: Minneota, Belgrade-Brooten-Elrosa battle...",https://www.startribune.com/minnesota-high-sch...
4,2026-11-14-18-41-10 +0000,nypost,Plan by NY lawmakers to blackball The Post cru...,https://nypost.com/2026/11/14/us-news/plan-by-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2690/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
48,trump,29
305,ground,24
243,nvidia,21
245,jensen,18
246,huang,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
168,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...,118
54,2026-11-14-15-46-18 +0000,nypost,Steamy photos show ex-Nvidia boss Jensen Huang...,https://nypost.com/2026/11/14/us-news/huang-an...,98
29,2026-11-14-17-27-50 +0000,nypost,Musk wants disgraced ex-Nvidia boss Jensen Hua...,https://nypost.com/2026/11/14/world-news/elon-...,95
188,2026-11-14-00-49-44 +0000,latimes,Democrats press Trump administration for answe...,https://www.latimes.com/politics/story/2026-11...,94
173,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...,93


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
168,118,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...
201,73,2026-11-13-23-52-22 +0000,latimes,"LAX, other airports filled with stranded passe...",https://www.latimes.com/california/story/2026-...
225,65,2026-11-13-22-32-23 +0000,nyt,Lauren Boebert Loses Re-election Bid in Colora...,https://www.nytimes.com/2026/11/09/us/lauren-b...
54,63,2026-11-14-15-46-18 +0000,nypost,Steamy photos show ex-Nvidia boss Jensen Huang...,https://nypost.com/2026/11/14/us-news/huang-an...
140,54,2026-11-14-06-42-27 +0000,bbc,Microsoft says services fully restored after d...,https://www.bbc.co.uk/news/articles/c6rrg51rkz...
267,52,2026-11-13-21-06-14 +0000,nypost,"Trump will have historic meeting with Xi, Puti...",https://nypost.com/2026/11/13/us-news/trump-wi...
170,50,2026-11-14-02-03-00 +0000,wsj,California voters had a chance to impose the n...,https://www.wsj.com/politics/california-voters...
281,50,2026-11-13-20-29-03 +0000,wapo,"Before he was Pope Leo, he was their bishop. N...",https://www.washingtonpost.com/world/2026/11/1...
7,45,2026-11-14-18-35-00 +0000,wsj,Transportation Secretary Sean Duffy blames Con...,https://www.wsj.com/politics/policy/duffy-blam...
266,43,2026-11-13-21-06-41 +0000,cbc,'Miracle on the Mississippi': All aboard river...,https://www.cbc.ca/news/world/tanker-hits-crui...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
